# 04b LLM pipeline 1: structured JSON to a natural language explanation

The LLM receives global feature importance and local SHAP/EBM contributions as
structured JSON and writes a German explanation for non technical users.

* Advantage: precise, machine readable input.
* Disadvantage: no visual context (no plots).

For each combination (XGBoost/EBM x 10 instances) one explanation is generated
and saved in results/pipeline04/.

In [ ]:
from __future__ import annotations

import sys, json, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from utils import (
    INSTANCE_IDS,
    EXPLANATIONS_DIR, RESULTS_DIR, PROMPTS_DIR,
)
from utils.llm import ask_text, DEFAULT_MODEL, MAX_TOKENS_GENERATION, strip_scratchpad

LOSS_KEY   = "poisson_log"
MODEL      = DEFAULT_MODEL
MAX_TOKENS = MAX_TOKENS_GENERATION

# n=20 validity run: the 10 instances, 1 generation, real time,
# output directly into results/pipeline04/
GEN_INSTANCE_IDS = INSTANCE_IDS
N_GEN            = 1
OUT_DIR          = RESULTS_DIR / "pipeline04"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"LLM model:     {MODEL}")
print(f"Loss option:   {LOSS_KEY}")
print(f"Instances:     {len(GEN_INSTANCE_IDS)} x {N_GEN} generation")
print(f"Output:        {OUT_DIR}")

## 1. System prompt

The system prompt is cached (Anthropic prompt caching). Since the minimum for a
cache block is 1024 tokens, the system prompt contains, besides the instructions,
the full feature schema with descriptions, so it reliably crosses the threshold and
is read from the cache on all later calls.

In [ ]:
SYSTEM_PROMPT = (PROMPTS_DIR / "pipeline_04_json.md").read_text()

print(f"System prompt: {len(SYSTEM_PROMPT)} characters, ~{len(SYSTEM_PROMPT)//4} tokens (estimated)")

## 2. Helper functions

build_context_string turns normalised raw values into readable values
(for example temp=0.68 to ~27.9 C). This context line is attached to the JSON
payload as human_readable_context so the model does not have to denormalise itself.

In [ ]:
from utils.explanations import build_context_string
from utils import load_global_explanation, load_local_explanation

# load_global_explanation / load_local_explanation: central IO helpers
# (utils.generation), path and loss schema in one place, tested.
# build_user_prompt stays visible: it defines what the LLM receives as JSON.

def build_user_prompt(global_exp: dict, local_exp: dict, top_k: int = 5) -> str:
    fv = local_exp["feature_values"]
    payload = {
        "model": local_exp["model"],
        "metrics": {
            "rmse":             global_exp["metrics"]["rmse"],
            "r2":               global_exp["metrics"]["r2"],
            "poisson_deviance": global_exp["metrics"]["poisson_deviance"],
        },
        "global_top_features": [
            {"rank": e["rank"], "feature": e["feature"]}
            for e in global_exp["global_importance"][:top_k]
        ],
        "instance_id":           local_exp["instance_id"],
        "feature_values":        fv,
        "human_readable_context": build_context_string(fv),
        "y_true":                local_exp["y_true"],
        "prediction":            local_exp["prediction"],
        "top_contributions":     local_exp["contributions"][:6],
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)


# Show an example prompt
g = load_global_explanation("xgb", loss_key=LOSS_KEY, explanations_dir=EXPLANATIONS_DIR)
l = load_local_explanation("xgb", INSTANCE_IDS[0], loss_key=LOSS_KEY, explanations_dir=EXPLANATIONS_DIR)
sample_prompt = build_user_prompt(g, l)
print(f"Example user prompt ({len(sample_prompt)} characters):")
print(sample_prompt)

## 3. LLM calls

In [ ]:
from utils import run_resumable_generation, build_generation_record
from utils.llm import build_text_params, run_params

# n=20 validity: real time over the central resume loop (skip if exists,
# idempotent, lossless, tested in tests/test_generation_loop.py). The persisted
# record is built via utils.build_generation_record (one schema for all
# pipelines, golden test in test_generation_loop.py).

def generate_json(model_name, iid, gen_idx):
    g_exp = load_global_explanation(model_name, loss_key=LOSS_KEY, explanations_dir=EXPLANATIONS_DIR)
    l_exp = load_local_explanation(model_name, iid, loss_key=LOSS_KEY, explanations_dir=EXPLANATIONS_DIR)
    params = build_text_params(
        build_user_prompt(g_exp, l_exp),
        system=SYSTEM_PROMPT, model=MODEL, max_tokens=MAX_TOKENS, cache_system=True,
    )
    t0 = time.time()
    response = run_params(params)
    elapsed  = time.time() - t0

    text  = strip_scratchpad(response["content"][0]["text"])
    usage = response.get("usage", {})
    record = build_generation_record(
        pipeline="04_json", model_name=model_name, instance_id=iid,
        explanation=text, usage=usage, llm_model=MODEL, loss_key=LOSS_KEY,
        prediction=l_exp["prediction"], y_true=l_exp["y_true"],
        elapsed_s=round(elapsed, 2),
    )
    u = record["usage"]
    print(f"  {model_name.upper()} inst={iid:4d} g{gen_idx}  "
          f"pred={record['prediction']:6.1f}  y={record['y_true']:5.0f}  "
          f"in={u['input_tokens']}  out={u['output_tokens']}  "
          f"cache={u['cache_read_input_tokens']}  t={elapsed:.1f}s")
    return record


results = run_resumable_generation(
    model_names=["xgb", "ebm"],
    instance_ids=GEN_INSTANCE_IDS,
    out_dir=OUT_DIR,
    generate=generate_json,
    n_generations=N_GEN,
)

totals = {
    "in":    sum(r["usage"]["input_tokens"] for r in results),
    "out":   sum(r["usage"]["output_tokens"] for r in results),
    "cache": sum(r["usage"]["cache_read_input_tokens"] for r in results),
}
print(f"\nTotal:  input={totals['in']}  output={totals['out']}  "
      f"cache_read={totals['cache']}  ({len(results)} units)")

## 4. Example explanations

In [ ]:
for rec in results[:2]:
    sep = '=' * 70
    print(sep)
    print(f"Model: {rec['xai_model'].upper()}  |  instance: {rec['instance_id']}  "
          f"|  prediction: {rec['prediction']:.1f}  |  actual: {rec['y_true']:.0f}")
    print(sep)
    print(rec["explanation"])
    print()

## 5. Summary

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        "Model":      r["xai_model"].upper(),
        "Instance":   r["instance_id"],
        "y_true":     r["y_true"],
        "Prediction": r["prediction"],
        "Words":      len(r["explanation"].split()),
        "tok_input":  r["usage"]["input_tokens"],
        "tok_output": r["usage"]["output_tokens"],
        "Time (s)":   r["elapsed_s"],
    }
    for r in results
])
display(summary)